# 🇲🇦 Pipeline Complet — Analyse de Sentiments (Darija Marocain)

Ce notebook couvre toutes les étapes :
1. **Imports & configuration** — charger les bibliothèques
2. **EDA** — explorer chaque fichier source
3. **Définition des fonctions** — nettoyage, déduplication, détection de script *(avant toute utilisation)*
4. **Fusion** — normaliser et concatener les 7 datasets
5. **Nettoyage du texte** — arabizi, bruit web, diacritiques, normalisation, stop words
6. **Modèles classiques** — TF-IDF + SVM / Naive Bayes / Random Forest
7. **Deep Learning** — CNN, BiLSTM, modèle hybride, attention
8. **DarijaBERT** — fine-tuning du transformer pré-entraîné


## 1. Imports & Configuration
> On charge toutes les bibliothèques en une seule fois pour éviter les imports dispersés.

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

# ── Graine globale pour la reproductibilité ──
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("✓ Imports terminés")

✓ Imports terminés


## 2. Montage Google Drive & Chemins des Fichiers
> On monte le Drive une seule fois et on déclare tous les chemins ici.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Répertoire racine du projet ──
BASE_PATH = "/content/drive/MyDrive/analyse_de_sentiments"

files = {
    "moroccan_dialectal": os.path.join(BASE_PATH, "Moroccan_Dialectal_Reviews_dataset.csv"),
    "afrisenti":          os.path.join(BASE_PATH, "afrisenti-semeval_morrocan.csv"),
    "sans_standard":      os.path.join(BASE_PATH, "dataset_sans_standard.csv"),
    "hugging_face":       os.path.join(BASE_PATH, "hugging_face_review_label.csv"),
    "data_cleaned":       os.path.join(BASE_PATH, "DATA_CLEANED.csv"),
    "ElecMorocco2016_mda":os.path.join(BASE_PATH, "ElecMorocco2016_mda_.csv"),
    "Question":           os.path.join(BASE_PATH, "lyte_questions_seulement.csv"),
}

print("✓ Chemins configurés")

Mounted at /content/drive
✓ Chemins configurés


## 3. Exploration des Données (EDA)
> On inspecte rapidement chaque fichier : shape, colonnes, aperçu.

In [ ]:
def eda_light(file_path):
    """Affiche un résumé rapide d'un fichier CSV."""
    print("=" * 70)
    print(f"Fichier : {os.path.basename(file_path)}")
    print("=" * 70)
    df = pd.read_csv(file_path)

    print(f"\nShape : {df.shape}")
    print(f"Colonnes : {df.columns.tolist()}")
    print("\nAperçu :")
    print(df.head(3))

    # Détection automatique de la première colonne textuelle
    text_col = next((c for c in df.columns if df[c].dtype == "object"), None)
    print(f"\nColonne texte détectée : {text_col}\n")

    return df, text_col


# ── Application sur tous les fichiers ──
eda_results = {}
for nom, path in files.items():
    df_temp, col_texte = eda_light(path)
    eda_results[nom] = {"df": df_temp, "text_col": col_texte}

Fichier : Moroccan_Dialectal_Reviews_dataset.csv

Shape : (600, 4)
Colonnes : ['Text', 'product', 'sentiment', 'Class']

Aperçu :
                                                Text       product sentiment  \
0  ماكينة عجباتني كاتحيد شعر من الجدر ومكاتخليش ل...  hair trimmer  Positive   
1   واعرة كاتشد الشارج لمدة طويلة وساهلة فالخدمة ...  hair trimmer  Positive   
2  زوينة وصغيرة ومطيورة كيما كايقولو هههه، فلوسها...  hair trimmer  Positive   

   Class  
0      1  
1      1  
2      1  

Colonne texte détectée : Text

Fichier : afrisenti-semeval_morrocan.csv

Shape : (6998, 2)
Colonnes : ['commentaire', 'sentiment']

Aperçu :
                                         commentaire sentiment
0  #nada0074 Jomo3a mobaraka inchallah 3liya we 3...  Positive
1  @nohita123 @Anyssa_Ch la daba homa li ghadi yk...   Neutral
2  @sansuuna matbkhlich 3lina wakha ma3rt fin kat...   Neutral

Colonne texte détectée : commentaire

Fichier : dataset_sans_standard.csv

Shape : (5476, 3)
Colonnes : ['twee

## 4. Détection du Script (Arabe / Latin / Mixte)
> Utile pour comprendre la composition linguistique de chaque dataset avant de nettoyer.

In [ ]:
def detect_script(text):
    """
    Classe le texte selon le script utilisé :
    - darija_ar : uniquement caractères arabes
    - latin     : uniquement caractères latins
    - mix       : arabe + latin (arabizi)
    - unknown   : aucun caractère reconnu
    """
    if pd.isna(text):
        return "unknown"
    text = str(text)
    has_arabic = bool(re.search(r'[\u0600-\u06FF]', text))
    has_latin  = bool(re.search(r'[a-zA-Z]', text))

    if has_arabic and not has_latin:   return "darija_ar"
    elif has_latin and not has_arabic: return "latin"
    elif has_arabic and has_latin:     return "mix"
    else:                              return "unknown"


for nom, data in eda_results.items():
    print("\n" + "=" * 70)
    print(f"DATASET : {nom}")
    print("=" * 70)
    df = data["df"]
    text_col = data["text_col"]
    df["script_type"] = df[text_col].apply(detect_script)
    print("\nDistribution des scripts :")
    print(df["script_type"].value_counts())


DATASET : moroccan_dialectal

Distribution des scripts :
script_type
darija_ar    591
mix            9
Name: count, dtype: int64

DATASET : afrisenti

Distribution des scripts :
script_type
latin    6809
mix       189
Name: count, dtype: int64

DATASET : sans_standard

Distribution des scripts :
script_type
darija_ar    5470
unknown         6
Name: count, dtype: int64

DATASET : hugging_face

Distribution des scripts :
script_type
darija_ar    573
latin        218
mix           60
Name: count, dtype: int64

DATASET : data_cleaned

Distribution des scripts :
script_type
darija_ar    13125
latin         4579
mix           1527
unknown        760
Name: count, dtype: int64

DATASET : ElecMorocco2016_mda

Distribution des scripts :
script_type
darija_ar    3716
mix           179
Name: count, dtype: int64

DATASET : Question

Distribution des scripts :
script_type
darija_ar    1982
mix            44
Name: count, dtype: int64


## 5. Distribution des Labels par Dataset
> On vérifie l'équilibre des classes avant de fusionner.

In [ ]:
def afficher_distribution_labels(df, col_label, nom):
    """Affiche les valeurs uniques et la distribution de la colonne label."""
    print("=" * 60)
    print(f" {nom}")
    print("=" * 60)
    print(f"  Colonne label   : '{col_label}'")
    print(f"  Valeurs uniques : {df[col_label].unique().tolist()}")
    print("  Distribution :")
    print(df[col_label].value_counts().to_string())
    print()


afficher_distribution_labels(pd.read_csv(files["moroccan_dialectal"]), "sentiment", "1. Moroccan Dialectal Reviews")
afficher_distribution_labels(pd.read_csv(files["afrisenti"]),           "sentiment", "2. AfriSenti SemEval")
afficher_distribution_labels(pd.read_csv(files["sans_standard"]),       "type",      "3. Dataset sans standard")
afficher_distribution_labels(pd.read_csv(files["hugging_face"]),        "label",     "4. Hugging Face Reviews")
afficher_distribution_labels(pd.read_csv(files["data_cleaned"]),        "polarity",  "5. DATA_CLEANED")
afficher_distribution_labels(pd.read_csv(files["ElecMorocco2016_mda"]), "sentiment", "6. ElecMorocco2016_mda")

 1. Moroccan Dialectal Reviews
  Colonne label   : 'sentiment'
  Valeurs uniques : ['Positive', 'Negative']
  Distribution :
sentiment
Positive    325
Negative    275

 2. AfriSenti SemEval
  Colonne label   : 'sentiment'
  Valeurs uniques : ['Positive', 'Neutral', 'Negative', nan]
  Distribution :
sentiment
Neutral     2919
Negative    1639
Positive    1490

 3. Dataset sans standard
  Colonne label   : 'type'
  Valeurs uniques : ['neutral', 'positive', 'negative', 'mixed']
  Distribution :
type
positive    2997
neutral     1311
negative     985
mixed        183

 4. Hugging Face Reviews
  Colonne label   : 'label'
  Valeurs uniques : ['negative', 'neutral', 'positive', 'negative ']
  Distribution :
label
positive     456
negative     269
neutral      122
negative       4

 5. DATA_CLEANED
  Colonne label   : 'polarity'
  Valeurs uniques : [1, -1]
  Distribution :
polarity
 1    9999
-1    9992

 6. ElecMorocco2016_mda
  Colonne label   : 'sentiment'
  Valeurs uniques : ['N', 'P']
  D

## 6. Définition des Fonctions de Nettoyage

>  **Important** : toutes les fonctions sont définies ICI, avant d'être appelées dans la fusion et le nettoyage.


In [ ]:
# ── 6.1 Arabizi → Arabe ─────────────────────────────────────────────────────
# L'arabizi est l'écriture du dialecte marocain avec des chiffres remplaçant
# des lettres arabes (ex. 7→ح, 3→ع). On convertit ces chiffres pour que les
# modèles arabes puissent les traiter correctement.

ARABIZI_MAP = {
    '7': 'ح', '3': 'ع', '9': 'ق',
    '2': 'أ', '5': 'خ', '8': 'غ',
}

def convert_arabizi_to_arabic(text):
    """
    Convertit les chiffres arabizi → lettres arabes dans les mots
    qui contiennent aussi des lettres (arabe ou latin).
    Exemples : 7rib → حrib | 3lash → علash | 9raya → قraya
    """
    text = str(text)
    words = text.split()
    result = []
    for word in words:
        has_letter  = bool(re.search(r'[a-zA-Z\u0600-\u06FF]', word))
        has_arabizi = bool(re.search(r'[235689]', word))
        if has_letter and has_arabizi:
            for digit, arabic_letter in ARABIZI_MAP.items():
                word = word.replace(digit, arabic_letter)
        result.append(word)
    return ' '.join(result)


# ── 6.2 Suppression du bruit web ────────────────────────────────────────────
# Les commentaires issus des réseaux sociaux contiennent souvent des @mentions,
# des URLs, des hashtags et des balises HTML qu'il faut supprimer.

def clean_web_noise(text):
    """Supprime mentions, URLs, hashtags, balises HTML et caractères invisibles."""
    text = str(text)
    text = re.sub(r'@\w+', '', text)                     # mentions
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # URLs
    text = re.sub(r'#\w+', '', text)                      # hashtags
    text = re.sub(r'<br\s*/?>', ' ', text)                # balises <br>
    text = re.sub(r'<[^>]+>', '', text)                   # autres balises HTML
    text = re.sub(r'[\u200c\u200d\uFEFF]', '', text)     # caractères invisibles
    return re.sub(r'\s+', ' ', text).strip()


# ── 6.3 Suppression des diacritiques arabes ──────────────────────────────────
# Les diacritiques (tashkeel) varient d'un scripteur à l'autre mais ne changent
# pas le sens ; les supprimer réduit le bruit dans le vocabulaire.

def remove_diacritics(text):
    """Supprime les diacritiques et signes de vocalisation arabes."""
    return re.sub(r'[\u064B-\u0652\u0670\uFE70-\uFEFF]', '', str(text))


# ── 6.4 Normalisation des caractères arabes ──────────────────────────────────
# Plusieurs formes de la même lettre existent en unicode (أ/إ/آ/ا → ا, ى → ي, etc.).
# On unifie pour éviter des tokens dupliqués.

def normalize_arabic(text):
    """Unifie les variantes de lettres arabes et supprime les tatweel."""
    text = str(text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    return re.sub(r'ـ+', '', text)    # supprime le tatweel (ـ)


# ── 6.5 Normalisation des répétitions ───────────────────────────────────────
# "زووووين" → "زوين" | "waaaaw" → "waw"
# Les gens répètent les lettres pour insister ; on réduit à une seule occurrence.

def normalize_repeated_chars(text):
    """Réduit les répétitions de 3+ caractères consécutifs à 1 seul."""
    text = str(text)
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text)
    text = re.sub(r'([\u0600-\u06FF])\1{2,}', r'\1', text)
    return text


# ── 6.6 Suppression des mots répétés ────────────────────────────────────────
def remove_repeated_words(text):
    """Supprime les mots dupliqués successifs ('bravo bravo bravo' → 'bravo')."""
    return re.sub(r'\b(\w+)( \1\b)+', r'\1', str(text))


# ── 6.7 Nettoyage ponctuation & chiffres ────────────────────────────────────
# Les chiffres isolés n'ont généralement pas de valeur sémantique pour l'analyse
# de sentiments ; on les supprime ainsi que la ponctuation superflue.

def clean_punctuation_numbers(text):
    """Supprime les chiffres, normalise la ponctuation répétée."""
    text = str(text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[()\[\]{}<>\"\'~`;,:#@%&*\-+/=\\]', ' ', text)
    text = re.sub(r'!{2,}', '!', text)
    text = re.sub(r'\?{2,}', '?', text)
    text = re.sub(r'\.{2,}', '.', text)
    return re.sub(r'\s+', ' ', text).strip()


# ── 6.8 Filtre de validité ───────────────────────────────────────────────────
def is_valid_text(text):
    """Retourne True si le texte contient au moins 2 mots (non vide/NaN)."""
    if pd.isna(text):
        return False
    return len(str(text).strip().split()) >= 2


# ── 6.9 Déduplication ───────────────────────────────────────────────────────
# Les datasets provenant de sources différentes peuvent contenir les mêmes
# commentaires. On supprime les doublons en comparant les textes normalisés
# (minuscules, espaces réduits) pour capturer les variantes mineures.

def supprimer_doublons(df: pd.DataFrame, text_col: str = "texte") -> pd.DataFrame:
    """
    Supprime les doublons en comparant une clé normalisée (minuscules + strip).
    Affiche le taux de duplication et des exemples.

    Paramètres
    ----------
    df       : DataFrame source
    text_col : nom de la colonne texte

    Retourne
    --------
    DataFrame dédupliqué, index réinitialisé.
    """
    cle       = df[text_col].astype(str).str.strip().str.lower()
    nb_doublons = cle.duplicated(keep="first").sum()
    avant       = len(df)

    print("=" * 60)
    print("ANALYSE DES DOUBLONS")
    print("=" * 60)
    print(f"  Total lignes      : {avant}")
    print(f"  Doublons détectés : {nb_doublons}")
    print(f"  Taux              : {nb_doublons / avant * 100:.2f}%")

    if nb_doublons > 0:
        masque = cle.duplicated(keep=False)
        exemples = df[masque].groupby(cle[masque]).first().head(3)
        print("\n  Exemples supprimés :")
        for i, (_, row) in enumerate(exemples.iterrows(), 1):
            extrait = str(row[text_col])[:80]
            print(f"  {i}. '{extrait}...'  [{row['label']}]")

    df_clean = df[~cle.duplicated(keep="first")].reset_index(drop=True)
    print(f"\n {avant} → {len(df_clean)} lignes  (−{avant - len(df_clean)})")
    return df_clean


print("✓ Toutes les fonctions de nettoyage sont définies")

✓ Toutes les fonctions de nettoyage sont définies


## 7. Fusion & Normalisation des Datasets

> On harmonise les colonnes (texte → `texte`, label → `label`) et on mappe
> tous les labels vers le format commun : **Positive / Negative / Neutral**.


In [ ]:
print("=" * 60)
print("NORMALISATION ET FUSION DES 7 DATASETS")
print("=" * 60)

tous_les_datasets = []

# ── Fichier 1 : Moroccan_Dialectal_Reviews ───────────────────────────────────
print("\n1. Moroccan_Dialectal_Reviews_dataset.csv")
df1 = pd.read_csv(files["moroccan_dialectal"])
df1_norm = pd.DataFrame({"texte": df1["Text"], "label": df1["sentiment"]})
tous_les_datasets.append(df1_norm)
print(f"  ✓ {len(df1_norm)} lignes")

# ── Fichier 2 : AfriSenti SemEval ────────────────────────────────────────────
# Les NaN dans la colonne sentiment sont remplacés par "Neutral"
print("\n2. afrisenti-semeval_morrocan.csv")
df2 = pd.read_csv(files["afrisenti"])
df2_norm = pd.DataFrame({
    "texte": df2["commentaire"],
    "label": df2["sentiment"].fillna("Neutral")
})
tous_les_datasets.append(df2_norm)
print(f"  ✓ {len(df2_norm)} lignes")

# ── Fichier 3 : Dataset sans standard ────────────────────────────────────────
# "Mixed" → "Neutral" car c'est une classe intermédiaire
print("\n3. dataset_sans_standard.csv")
df3 = pd.read_csv(files["sans_standard"])
df3_norm = pd.DataFrame({
    "texte": df3["tweets"],
    "label": (df3["type"]
              .str.strip()
              .str.capitalize()
              .replace("Mixed", "Neutral"))
})
tous_les_datasets.append(df3_norm)
print(f"  ✓ {len(df3_norm)} lignes")

# ── Fichier 4 : Hugging Face Reviews ─────────────────────────────────────────
print("\n4. hugging_face_review_label.csv")
df4 = pd.read_csv(files["hugging_face"])
df4_norm = pd.DataFrame({
    "texte": df4["review"],
    "label": df4["label"].str.strip().str.capitalize()
})
tous_les_datasets.append(df4_norm)
print(f"  ✓ {len(df4_norm)} lignes")

# ── Fichier 5 : DATA_CLEANED ──────────────────────────────────────────────────
# Les labels numériques (1=Positive, 0=Neutral, -1=Negative) sont convertis
print("\n5. DATA_CLEANED.csv")
df5 = pd.read_csv(files["data_cleaned"])
df5_norm = pd.DataFrame({
    "texte": df5["sentence"],
    "label": df5["polarity"].map({1: "Positive", 0: "Neutral", -1: "Negative"})
})
tous_les_datasets.append(df5_norm)
print(f"  ✓ {len(df5_norm)} lignes")

# ── Fichier 6 : ElecMorocco2016_mda_ ─────────────────────────────────────────
# Labels codés en lettres : P=Positive, N=Negative (pas de Neutral dans ce dataset)
print("\n6. ElecMorocco2016_mda_.csv")
df6 = pd.read_csv(files["ElecMorocco2016_mda"])
df6_norm = pd.DataFrame({
    "texte": df6["comment_message"],
    "label": df6["sentiment"].map({"P": "Positive", "N": "Negative"})
})
tous_les_datasets.append(df6_norm)
print(f"  ✓ {len(df6_norm)} lignes")

# ── Fichier 7 : lyte_questions_seulement ─────────────────────────────────────
# "Neutre" (français) → "Neutral" pour uniformisation
print("\n7. lyte_questions_seulement.csv")
df7 = pd.read_csv(files["Question"])
df7_norm = pd.DataFrame({
    "texte": df7["question"],
    "label": df7["sentiment"].map({
        "Positive": "Positive",
        "Negative": "Negative",
        "Neutre":   "Neutral"
    })
})
# Fallback : si le mapping retourne NaN, on capitalise la valeur brute
df7_norm["label"] = df7_norm["label"].fillna(df7["sentiment"].str.capitalize())
tous_les_datasets.append(df7_norm)
print(f"  ✓ {len(df7_norm)} lignes")

# ── Concaténation finale ──────────────────────────────────────────────────────
dataset_fusionne = pd.concat(tous_les_datasets, ignore_index=True)

# Suppression des lignes dont le label est NaN après mapping
avant = len(dataset_fusionne)
dataset_fusionne = dataset_fusionne.dropna(subset=["label"]).reset_index(drop=True)
print(f"\n  Labels NaN supprimés : {avant - len(dataset_fusionne)}")

print(f"\n{'='*60}")
print(f"TOTAL FUSIONNÉ : {len(dataset_fusionne)} lignes")
print("\nDistribution des labels :")
print(dataset_fusionne["label"].value_counts())

# Sauvegarde intermédiaire pour référence
dataset_fusionne.to_csv("dataset_fusionner.csv", index=False, encoding="utf-8-sig")
print("\n Sauvegardé : dataset_fusionner.csv")

NORMALISATION ET FUSION DES 7 DATASETS

1. Moroccan_Dialectal_Reviews_dataset.csv
  ✓ 600 lignes

2. afrisenti-semeval_morrocan.csv
  ✓ 6998 lignes

3. dataset_sans_standard.csv
  ✓ 5476 lignes

4. hugging_face_review_label.csv
  ✓ 851 lignes

5. DATA_CLEANED.csv
  ✓ 19991 lignes

6. ElecMorocco2016_mda_.csv
  ✓ 3895 lignes

7. lyte_questions_seulement.csv
  ✓ 2026 lignes

  Labels NaN supprimés : 0

TOTAL FUSIONNÉ : 39837 lignes

Distribution des labels :
label
Positive    16530
Negative    15796
Neutral      7511
Name: count, dtype: int64

 Sauvegardé : dataset_fusionner.csv


## 8. Déduplication & Pipeline de Nettoyage Complet

> On applique dans l'ordre :
> 1. Déduplication  
> 2. Arabizi → Arabe  
> 3. Suppression du bruit web  
> 4. Suppression des diacritiques  
> 5. Normalisation des caractères arabes  
> 6. Filtre de validité (≥ 2 mots)


In [ ]:
# ── Étape 1 : Déduplication ──────────────────────────────────────────────────
df_final = supprimer_doublons(dataset_fusionne, text_col="texte")

# ── Étape 2-5 : Pipeline de nettoyage ───────────────────────────────────────
print("\nApplication du pipeline de nettoyage...")
df_final["texte_clean"] = df_final["texte"].apply(convert_arabizi_to_arabic)
df_final["texte_clean"] = df_final["texte_clean"].apply(clean_web_noise)
df_final["texte_clean"] = df_final["texte_clean"].apply(remove_diacritics)
df_final["texte_clean"] = df_final["texte_clean"].apply(normalize_arabic)
df_final["texte_clean"] = df_final["texte_clean"].apply(normalize_repeated_chars)
df_final["texte_clean"] = df_final["texte_clean"].apply(clean_punctuation_numbers)

# ── Étape 6 : Filtre de validité ─────────────────────────────────────────────
avant = len(df_final)
df_final = df_final[df_final["texte_clean"].apply(is_valid_text)].reset_index(drop=True)
print(f"  Textes trop courts supprimés : {avant - len(df_final)}")

# Sauvegarde
df_final.to_csv("dataset_clean.csv", index=False, encoding="utf-8-sig")
print(f"\n dataset_clean.csv sauvegardé : {len(df_final)} lignes")
print("\nDistribution finale des labels :")
print(df_final["label"].value_counts())

ANALYSE DES DOUBLONS
  Total lignes      : 39837
  Doublons détectés : 2927
  Taux              : 7.35%

  Exemples supprimés :
  1. ' ...'  [Positive]
  2. '"""2023💔"...'  [Negative]
  3. '"""2024 😢"...'  [Negative]

 39837 → 36910 lignes  (−2927)

Application du pipeline de nettoyage...
  Textes trop courts supprimés : 1161

 dataset_clean.csv sauvegardé : 35749 lignes

Distribution finale des labels :
label
Positive    14456
Negative    14340
Neutral      6953
Name: count, dtype: int64


## 9. Suppression des Stop Words

> Les stop words (mots très fréquents sans valeur sémantique : *wa*, *dyal*, *li*...)
> sont supprimés pour améliorer la représentation TF-IDF et réduire le bruit.


In [ ]:
# ── Chargement de la liste de stop words ─────────────────────────────────────
# Le fichier est en UTF-16 (encodage courant pour les fichiers arabes sous Windows)
sw_df = pd.read_csv(
    os.path.join(BASE_PATH, "Stop_words.csv"),
    encoding="utf-16", header=None,
    keep_default_na=False, dtype=str
)
stop_words_set = set(w.lower().strip() for w in sw_df.iloc[:, 0] if w.strip())
print(f"✓ Stop words chargés : {len(stop_words_set)} mots uniques")


def remove_stop_words(text, sw):
    """Supprime les stop words d'un texte (comparaison en minuscules)."""
    return " ".join(w for w in str(text).lower().split() if w not in sw)


def tokenize(text):
    """Tokenisation simple par espaces."""
    return str(text).strip().split()


# ── Application sur le dataset nettoyé ───────────────────────────────────────
df_clean = pd.read_csv("dataset_clean.csv")
df_clean["texte_ml"] = df_clean["texte_clean"].apply(lambda x: remove_stop_words(x, stop_words_set))
df_clean["tokens"]   = df_clean["texte_ml"].apply(tokenize)

# Supprimer les textes avec moins de 2 tokens après stop words
avant = len(df_clean)
df_clean = df_clean[df_clean["tokens"].apply(len) >= 2].reset_index(drop=True)
print(f"  Textes trop courts après stop words : {avant - len(df_clean)}")

print(f"\nShape final : {df_clean.shape}")
print(df_clean[["texte_ml", "tokens", "label"]].head(3))

# Sauvegarde du dataset préprocessé (utilisé par tous les modèles)
df_clean.to_csv("dataset_preprocessed.csv", index=False, encoding="utf-8-sig")
print("\n Sauvegardé : dataset_preprocessed.csv")

✓ Stop words chargés : 468 mots uniques
  Textes trop courts après stop words : 65

Shape final : (35684, 5)
                                            texte_ml  \
0  ماكينه عجباتني كاتحيد شعر من الجدر ومكاتخليش ل...   
1  واعره كاتشد الشارج لمده طويله وساهله فالخدمه د...   
2  زوينه وصغيره ومطيوره كيما كايقولو ه، فلوسها حل...   

                                              tokens     label  
0  [ماكينه, عجباتني, كاتحيد, شعر, من, الجدر, ومكا...  Positive  
1  [واعره, كاتشد, الشارج, لمده, طويله, وساهله, فا...  Positive  
2  [زوينه, وصغيره, ومطيوره, كيما, كايقولو, ه،, فل...  Positive  

 Sauvegardé : dataset_preprocessed.csv


## 10. Statistiques Finales du Dataset
> Résumé avant la modélisation.

In [ ]:
df_stats = pd.read_csv("dataset_preprocessed.csv")

print("=" * 60)
print("RÉSUMÉ DU DATASET PRÉPROCESSÉ")
print("=" * 60)
print(f"  Taille totale       : {len(df_stats)} lignes")
print(f"  Labels uniques      : {df_stats['label'].unique().tolist()}")
print("\nDistribution des labels :")
print(df_stats["label"].value_counts())
print("\nExemple de texte nettoyé :")
print(df_stats[["texte", "texte_clean", "texte_ml", "label"]].head(5))

RÉSUMÉ DU DATASET PRÉPROCESSÉ
  Taille totale       : 35684 lignes
  Labels uniques      : ['Positive', 'Negative', 'Neutral']

Distribution des labels :
label
Positive    14406
Negative    14329
Neutral      6949
Name: count, dtype: int64

Exemple de texte nettoyé :
                                               texte  \
0  ماكينة عجباتني كاتحيد شعر من الجدر ومكاتخليش ل...   
1   واعرة كاتشد الشارج لمدة طويلة وساهلة فالخدمة ...   
2  زوينة وصغيرة ومطيورة كيما كايقولو هههه، فلوسها...   
3   عاد جابوهاليا التوصيل كان فالمستوى والمنتوج و...   
4                         👌🏻 رهييييب حرفياً انصح فيه   

                                         texte_clean  \
0  ماكينه عجباتني كاتحيد شعر من الجدر ومكاتخليش ل...   
1  واعره كاتشد الشارج لمده طويله وساهله فالخدمه د...   
2  زوينه وصغيره ومطيوره كيما كايقولو ه، فلوسها حل...   
3  عاد جابوهاليا التوصيل كان فالمستوي والمنتوج وص...   
4                             👌🏻 رهيب حرفيا انصح فيه   

                                            texte_ml     l

## 11. Modèles Classiques (TF-IDF + ML)

> On utilise TF-IDF (représentation statistique des mots) comme features d'entrée
> pour trois modèles classiques : SVM, Naive Bayes, Random Forest.


In [ ]:
# ── Chargement et préparation ────────────────────────────────────────────────
df = pd.read_csv("dataset_preprocessed.csv")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print(f"Mapping des labels : {label_mapping}")

X = df["texte_ml"].values   # texte déjà nettoyé sans stop words
y = df["label_encoded"].values

# ── Split stratifié 70/15/15 ─────────────────────────────────────────────────
# Stratifié = on maintient la même proportion de classes dans chaque split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"  Train      : {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation : {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test       : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

Mapping des labels : {'Negative': 0, 'Neutral': 1, 'Positive': 2}
  Train      : 24978 (70.0%)
  Validation : 5353 (15.0%)
  Test       : 5353 (15.0%)


In [ ]:
# ── TF-IDF Vectorisation ──────────────────────────────────────────────────────
# max_features=15000 : on garde les 15 000 n-grammes les plus informatifs
# ngram_range=(1,3)  : unigrams, bigrams et trigrams
# sublinear_tf=True  : remplace tf par 1+log(tf) pour atténuer les très hautes fréquences

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True,
    analyzer='word'
)

X_train_tfidf = tfidf.fit_transform(X_train)  # fit uniquement sur train !
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Features TF-IDF : {X_train_tfidf.shape[1]}")
print(f"Train  : {X_train_tfidf.shape}")
print(f"Val    : {X_val_tfidf.shape}")
print(f"Test   : {X_test_tfidf.shape}")

Features TF-IDF : 15000
Train  : (24978, 15000)
Val    : (5353, 15000)
Test   : (5353, 15000)


In [ ]:
# ── Poids des classes (gestion du déséquilibre) ──────────────────────────────
# Les classes ne sont pas équilibrées : on donne plus de poids aux classes rares
# pour que le modèle ne les ignore pas.
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print(f"Poids des classes : {class_weight_dict}")


def evaluate_model(model, X_tr, y_tr, X_v, y_v, X_te, y_te, model_name):
    """Entraîne et évalue un modèle classique, affiche les métriques détaillées."""
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")

    model.fit(X_tr, y_tr)

    y_v_pred  = model.predict(X_v)
    y_te_pred = model.predict(X_te)

    val_acc  = accuracy_score(y_v, y_v_pred)
    test_acc = accuracy_score(y_te, y_te_pred)
    val_f1   = f1_score(y_v, y_v_pred, average='weighted')
    test_f1  = f1_score(y_te, y_te_pred, average='weighted')
    test_prec = precision_score(y_te, y_te_pred, average='weighted')
    test_rec  = recall_score(y_te, y_te_pred, average='weighted')

    print(f"  Validation → Accuracy: {val_acc:.4f}  F1: {val_f1:.4f}")
    print(f"  Test       → Accuracy: {test_acc:.4f}  F1: {test_f1:.4f}")
    print(f"  Precision : {test_prec:.4f}  Recall : {test_rec:.4f}")

    # Détection du surapprentissage
    gap = abs(test_acc - val_acc)
    if gap < 0.03:   print(f"   Pas de surapprentissage (gap={gap:.4f})")
    elif gap < 0.05: print(f"   Léger surapprentissage (gap={gap:.4f})")
    else:            print(f"   Surapprentissage significatif (gap={gap:.4f})")

    print("\n  Classification Report (Test):")
    print(classification_report(y_te, y_te_pred, target_names=label_encoder.classes_))

    return {"model": model_name, "val_acc": val_acc, "test_acc": test_acc,
            "val_f1": val_f1, "test_f1": test_f1, "y_pred": y_te_pred, "gap": gap}


ml_results = {}

# ── SVM ──────────────────────────────────────────────────────────────────────
# kernel='linear' : rapide et efficace sur les données textuelles
svm_model = SVC(kernel='linear', C=1.0, class_weight=class_weight_dict,
                random_state=RANDOM_SEED, probability=True)
ml_results["SVM"] = evaluate_model(
    svm_model, X_train_tfidf, y_train, X_val_tfidf, y_val, X_test_tfidf, y_test,
    "Support Vector Machine"
)


Poids des classes : {0: np.float64(0.8301096709870389), 1: np.float64(1.7117598684210527), 2: np.float64(0.8256644188813963)}

  Support Vector Machine
  Validation → Accuracy: 0.7052  F1: 0.7063
  Test       → Accuracy: 0.7073  F1: 0.7082
  Precision : 0.7124  Recall : 0.7073
   Pas de surapprentissage (gap=0.0021)

  Classification Report (Test):
              precision    recall  f1-score   support

    Negative       0.74      0.70      0.72      2150
     Neutral       0.61      0.74      0.67      1042
    Positive       0.73      0.70      0.72      2161

    accuracy                           0.71      5353
   macro avg       0.70      0.71      0.70      5353
weighted avg       0.71      0.71      0.71      5353

